In [ ]:
import re

import numpy as np
import pandas as pd
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

In [ ]:
# Path to the token-level corpus data
PATH = "" # all tokens from corpus

# Gender labels used in the dataset
FEMALE = 0
MALE = 1


def extract_lemma(morph, word=None):
    """
    Extract the lemma from a morphological annotation.
    Falls back to the original token if the lemma is unavailable.
    """
    if pd.isna(morph):
        return str(word).lower() if word is not None else None

    morph = str(morph).strip()
    match = re.search(r"^\s*([^+\s]+)\+", morph)

    if match:
        return match.group(1).lower()

    return str(word).lower() if word is not None else None


def extract_pos(morph):
    """
    Extract the POS tag from a morphological annotation.
    """
    if pd.isna(morph):
        return None

    match = re.search(r"//_([^_]+)_", str(morph))
    return match.group(1) if match else None


# Load the corpus
df = pd.read_csv(PATH)

# Extract lemmas and POS tags
df["lemma"] = [
    extract_lemma(morph, word)
    for morph, word in zip(df["morph"], df["word"])
]
df["pos"] = df["morph"].apply(extract_pos)

# Keep only rows with complete metadata
df = df.dropna(subset=["lemma", "ID", "gender"])

# Normalize lemma strings
df["lemma"] = df["lemma"].str.lower().str.strip()

# Remove empty and numeric-only tokens
df = df[df["lemma"] != ""]
df = df[~df["lemma"].str.fullmatch(r"\d+")]

In [ ]:
# Minimum frequency and speaker coverage required for each lemma
MIN_TOTAL = 20
MIN_SPEAKERS = 5

# Calculate total frequency and number of unique speakers per lemma
lemma_total = df.groupby("lemma").size()
lemma_speakers = df.groupby("lemma")["ID"].nunique()

# Keep lemmas that meet both thresholds
valid_lemmas = lemma_total[
    (lemma_total >= MIN_TOTAL)
    & (lemma_speakers >= MIN_SPEAKERS)
].index

# Create a filtered dataset containing only valid lemmas
df_key = df[df["lemma"].isin(valid_lemmas)].copy()

In [ ]:
def keyness_analysis(data, female_code=0, male_code=1):
    """Compare lemma frequencies between female and male speakers."""

    # Total number of tokens in each group
    total_female = (data["gender"] == female_code).sum()
    total_male = (data["gender"] == male_code).sum()

    rows = []

    for lemma, sub in data.groupby("lemma"):
        female_count = (sub["gender"] == female_code).sum()
        male_count = (sub["gender"] == male_code).sum()

        # Construct a 2 × 2 contingency table:
        # lemma / all other tokens for each gender
        a = female_count
        b = total_female - female_count
        c = male_count
        d = total_male - male_count

        observed = np.array([a, b, c, d], dtype=float)

        row1 = a + b
        row2 = c + d
        col1 = a + c
        col2 = b + d
        grand = row1 + row2

        # Expected frequencies under the null hypothesis
        expected = np.array([
            row1 * col1 / grand,
            row1 * col2 / grand,
            row2 * col1 / grand,
            row2 * col2 / grand
        ])

        # Calculate the log-likelihood ratio (G²).
        # Zero observed frequencies contribute zero to the statistic.
        with np.errstate(divide="ignore", invalid="ignore"):
            terms = np.where(
                observed > 0,
                observed * np.log(observed / expected),
                0
            )

        G2 = 2 * terms.sum()
        p = chi2.sf(G2, df=1)

        # Normalize frequencies per 10,000 tokens
        female_per_10k = female_count / total_female * 10_000
        male_per_10k = male_count / total_male * 10_000

        # Positive values indicate higher relative frequency among women;
        # negative values indicate higher relative frequency among men.
        log_ratio = np.log2(
            ((female_count + 0.5) / total_female)
            / ((male_count + 0.5) / total_male)
        )

        rows.append({
            "lemma": lemma,
            "female_count": female_count,
            "male_count": male_count,
            "female_per_10k": female_per_10k,
            "male_per_10k": male_per_10k,
            "log_ratio_female_vs_male": log_ratio,
            "more_frequent_in": "female" if log_ratio > 0 else "male",
            "G2": G2,
            "p": p
        })

    result = pd.DataFrame(rows)

    # Correct p-values for multiple comparisons
    result["FDR"] = multipletests(
        result["p"],
        method="fdr_bh"
    )[1]

    # Prioritize significant lemmas with larger G² values
    result = result.sort_values(
        ["FDR", "G2"],
        ascending=[True, False]
    )

    return result

In [ ]:
# POS categories used for the style-focused analysis
STYLE_POS = {"P", "D", "I", "J"}

# Keep only tokens belonging to the selected POS categories
df_style = df[df["pos"].isin(STYLE_POS)].copy()

# Calculate frequency and speaker coverage for each lemma
lemma_total = df_style.groupby("lemma").size()
lemma_speakers = df_style.groupby("lemma")["ID"].nunique()

# Keep lemmas that meet the minimum thresholds
valid_style_lemmas = lemma_total[
    (lemma_total >= MIN_TOTAL)
    & (lemma_speakers >= MIN_SPEAKERS)
].index

df_style_key = df_style[
    df_style["lemma"].isin(valid_style_lemmas)
].copy()

# Run keyness analysis on the style-related lemmas
keyness_style = keyness_analysis(df_style_key, FEMALE, MALE)

# Save the results and preview the top-ranked lemmas
keyness_style.to_csv("keyness_style_lemmas.csv", index=False)

keyness_style.head(30)